In [229]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as smp
from sympy.polys.polyfuncs import interpolate
from matplotlib.pyplot import figure
from scipy.interpolate import CubicSpline
from scipy.interpolate import PchipInterpolator
import ipywidgets as widgets
from ipywidgets import interact
from scipy.interpolate import UnivariateSpline
from IPython.display import display, Math
from scipy.special import roots_legendre

# Задание 1. Двухточечная квадратура Гаусса.

Постройте квадратурную формулу Гаусса, интегрирующую точно многочлены степеней вплоть до третьей на интервале $[a, b]$. Заметим, что для этого достаточно построить двухточечную квадратуру.

Напишите функцию, которая возвращает веса, $\omega_1$ и $\omega_2$, и узлы, $x_1$ и $x_2$, двухточечной квадратурной формулы Гаусса с весовой функцией $w(x) = 1$, т.е., интегралы вида

$$ \int_{a}^{b} f(x) \, dx \approx w_1 f(x_1) + w_2 f(x_2) $$


In [35]:
def gauss_2(a, b):
    r"""Return nodes and weights for a two-point Gauss quadrature on [a, b].
    
    Parameters
    ----------
    a, b : float
       Boundaries of the integration interval
       
    Returns
    -------
    x1, x2, w1, w2 : float
       Nodes and weights of the quadrature.
    """
    roots, weights = roots_legendre(2)
    
    roots = 0.5*(b+a) + 0.5*(b-a)*roots
    
    weights *= (b-a)/2
    
    return np.concatenate((roots, weights))

In [37]:
from numpy.testing import assert_allclose

x1, x2, w1, w2 = gauss_2(0, 1)

def f(x, n): 
    return x**n

for n in [0, 1, 2, 3]:
    assert_allclose(w1*f(x1, n=n) + w2*f(x2, n=n),
                    1./(n+1), atol=1e-14)

# Задание 2. Вычитание интегрируемых сингулярностей.

Вычислите определённый интеграл методом трапеций с вычитанием сингулярности

$$ I = \int_{0}^{1} \frac{e^x}{\sqrt{x(1-x)}} \, dx $$

Вам могут пригодиться значения следующих определенных интегралов:

$$ \int_{0}^{1} \frac{1}{\sqrt{x(1-x)}} \, dx = \pi, \quad \int_{0}^{1} \frac{x}{\sqrt{x(1-x)}} \, dx = \pi/2 $$

Преобразуйте данный интеграл, вычитая сингулярности. Выпишите расчетные формулы. Cоставьте функцию, возвращающую значение интеграла методом трапеций. Проверьте точность ответа методом Рунге.

$$ I = \int_{0}^{\frac{1}{2}} \frac{e^x}{\sqrt{x(1-x)}} \, dx +  \int_{\frac{1}{2}}^{1} \frac{e^x}{\sqrt{x(1-x)}} \, dx = \int_{0}^{\frac{1}{2}} \frac{e^x - 1 - x}{\sqrt{x(1-x)}} + \int_{0}^{\frac{1}{2}} \frac{1 + x}{\sqrt{x(1-x)}} \, dx + \int_{\frac{1}{2}}^{1} \frac{e^x - e - (x-1)e}{\sqrt{x(1-x)}} + \int_{\frac{1}{2}}^{1} \frac{e + (x-1)e}{\sqrt{x(1-x)}}\, dx = \\ = 
\int_{0}^{\frac{1}{2}} \frac{e^x - 1 - x}{\sqrt{x(1-x)}} + \int_{\frac{1}{2}}^{1} \frac{e^x - ex}{\sqrt{x(1-x)}} + \int_{0}^{\frac{1}{2}} \frac{1 + x}{\sqrt{x(1-x)}} \, dx + \int_{\frac{1}{2}}^{1} \frac{ex}{\sqrt{x(1-x)}}\, dx $$

$$ \int_{0}^{1} \frac{1}{\sqrt{x(1-x)}}\, dx  = \left|  \xi = x - \frac{1}{2} \right | =  \int_{-\frac{1}{2}}^{\frac{1}{2}} \frac{1}{\sqrt{\left(\xi + \frac{1}{2}\right)\left(\frac{1}{2} - \xi\right)}}\, d\xi = 
\int_{-\frac{1}{2}}^{\frac{1}{2}} \frac{1}{\sqrt{\frac{1}{4} - \xi^2}}\, d\xi = 2 \int_{0}^{\frac{1}{2}} \frac{1}{\sqrt{\frac{1}{4} - \xi^2}}\, d\xi = 2 \int_{\frac{1}{2}}^{1} \frac{1}{\sqrt{x(1-x)}}\, dx \Rightarrow \\
\Rightarrow \int_{0}^{\frac{1}{2}} \frac{1}{\sqrt{x(1-x)}}\, dx = \int_{\frac{1}{2}}^{1} \frac{1}{\sqrt{x(1-x)}}\, dx = \frac{\pi}{2}
$$

$$
\int_{0}^{1} \frac{x}{\sqrt{x(1-x)}}\, dx = 
\int_{-\frac{1}{2}}^{\frac{1}{2}} \frac{\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi = 
\int_{0}^{\frac{1}{2}} \frac{\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi + \int_{-\frac{1}{2}}^{0} \frac{\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi = 
\int_{0}^{\frac{1}{2}} \frac{\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi + \int_{0}^{\frac{1}{2}} \frac{-\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi \\
\int_{\frac{1}{2}}^{1} \frac{x}{\sqrt{x(1-x)}}\, dx = 
\int_{0}^{\frac{1}{2}} \frac{\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi = 
\frac{1}{2}\int_{0}^{\frac{1}{4}} \frac{1}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi^2 + \frac{1}{2}\int_{0}^{\frac{1}{2}} \frac{1}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi =
\frac{1}{2} + \frac{1}{2} \cdot \frac{\pi}{2} = \frac{\pi + 2}{4}\\
\int_{0}^{\frac{1}{2}} \frac{x}{\sqrt{x(1-x)}}\, dx = 
\int_{0}^{\frac{1}{2}} \frac{-\xi + \frac{1}{2}}{\sqrt{\frac{1}{4} - \xi^2}} \, d\xi = \frac{\pi - 2}{4}
$$

$$
I = \int_{0}^{\frac{1}{2}} \frac{e^x - 1 - x}{\sqrt{x(1-x)}} + \int_{\frac{1}{2}}^{1} \frac{e^x - ex}{\sqrt{x(1-x)}} + \frac{3\pi - 2 + e(\pi + 2)}{4}
$$

Формула трапеций $\int_a^b f(x) dx = \frac{h}{2} \left( f(x_0) + 2f(x_1) + \cdots + 2f(x_{n-1}) + f(x_n) \right)$

In [157]:
def trapezoid(f, a, b, npts):
    h = (b-a)/(npts-1)
    
    points = np.linspace(a, b, npts)
    result = f(points[0]) + f(points[-1])
    
    if (npts > 2):
        result += 2*np.sum(np.vectorize(f)(points[1:-1]))
        
    result *= 0.5*h
    
    return result

In [158]:
def integ(npts=10):
    """Compute the value of the integral above.
    
    Subtract the singularities and use the trapezoid rule. 
    
    Parameters
    ----------
    npts : int
        The number of points for the trapezoid rule
        
    Returns
    -------
    I : float
       The computed value of the integral
    """
    
    def f(t):
        if t == 0:
            return 0
        return (np.e**t - 1 - t)/np.sqrt(t*(1-t))
    
    def g(t):
        if t == 1:
            return 0
        return (np.e**t - np.e*t)/np.sqrt(t*(1-t))
    
    I = trapezoid(f, 0, 0.5, npts) + trapezoid(g, 0.5, 1, npts) + (3*np.pi - 2 + np.e*(np.pi + 2))/4
    
    return I

In [166]:
print(integ())

5.5092788804924835


$$
n = 10 \Rightarrow h = 1/9; \quad h/2 = 1/18 \Rightarrow n' = 19 \\ \Delta_{h/2} = \frac{I_h - I_{h/2}}{2^p - 1}; \quad p = 2 \Rightarrow \Delta_{h/2} = \frac{I_h - I_{h/2}}{3} = \frac{integ(10) - integ(19)}{3}
$$

In [173]:
print('I =', integ(19), '+-', (integ(10) - integ(19))/3)

I = 5.508644638974998 +- 0.00021141383916193726


# Задание 4. Вычисление интеграла с внутренней особенностью.

Найти аналитически левую и правую окрестности нуля $\delta_1$ и $\delta_2$ такие, чтобы при вычислении интеграла

$$ J = \int_{-0.5}^{0.5} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx $$

модуль $|\rho| < \varepsilon$, где $\varepsilon - $ требуемая точность расчетов (см. теорию ранее).

$$ 
\int_{0}^{\delta} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx = \int_{0}^{\delta} \frac{dx}{\sqrt{x(1-x)}} \\
x < \frac{1}{2} \Rightarrow 1-x > \frac{1}{2} \Rightarrow \frac{1}{\sqrt{x(1-x)}} < \frac{\sqrt{2}}{\sqrt{x}} \Rightarrow \int_{0}^{\delta} \frac{dx}{\sqrt{x(1-x)}} < \sqrt{2}\int_{0}^{\delta} \frac{dx}{\sqrt{x}} = 2\sqrt{2} \sqrt{\delta} < \varepsilon \Rightarrow \delta < \frac{\varepsilon^2}{8}
$$

$$ 
\int_{-\delta}^{0} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx = \int_{0}^{\delta} \frac{dx}{\sqrt{x(1+x)}} \\
1 + x > \frac{1}{2} \quad \text{Можно применить ту же оценку} \quad \delta < \frac{\varepsilon^2}{8}
$$

$$
|\rho| = \left|\int_{-\delta_1}^{0} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx + \int_{0}^{\delta_2} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx \right| < \varepsilon \\
\text{Пусть} \quad \int_{-\delta_1}^{0} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx , \int_{0}^{\delta_2} |x|^{-\frac{1}{2}} \cdot (1 - x)^{-\frac{1}{2}} \, dx < \frac{\varepsilon}{2}. \quad \text{Выполняется при} \quad \delta_1, \delta_2 < \frac{\varepsilon^2}{32}
$$

# Задание 5. Интеграл от быстроосциллирующей функции.

Вычислите интеграл от быстроосциллирующей функции

$$ \int_{0}^{\pi} e^{-x} \sin kx \, dx $$

для различных значений $k$ по методу Симпсона, а затем используя прием, описанный ранее, вычитая из $e^{-x}$ её аппроксимацю полиномом второй степени (третьей степени). Совпадает-ли точность расчетов с ожидаемой? Сравните с точным значением интеграла, который равен

$$ \int_{0}^{\pi} e^{-x} \sin kx \, dx = \frac{k}{1 + k^2} - \frac{e^{-\pi} (k \cos k\pi + \sin k\pi)}{1 + k^2} $$

Метод Симпсона $\int_a^b f(x) \, dx = \frac{h}{3} (f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 4f(x_{n-3}) + 2f(x_{n-2}) + 4f(x_{n-1}) + f(x_n)) \\$
Погрешность метода $\frac{(b-a)h^4}{180} M_4$

$\quad \left(e^{-x} \sin kx\right)^{(4)} = e^{-x} \left( (k^4 - 6k^2 + 1) \sin(kx) + 4k(k^2 - 1) \cos(kx) \right) ; \quad M_4 < k^4 + 4k^3 - 6k^2 - 4k + 1, \quad k > 3
$

Погрешность метода Симпсона $ < \frac{\pi h^4}{180} (k^4 + 4k^3 - 6k^2 - 4k + 1) $

In [358]:
def simpson(f, a, b, npts):
    h = (b-a)/(npts-1)
    
    points = np.linspace(0, np.pi, npts)
    
    f_vect = np.vectorize(f)
    
    result = f(points[0]) + f(points[-1]) + 4*np.sum(f_vect(points[1:-1:2])) + 2*np.sum(f_vect(points[2:-2:2]))
    
    result *= h/3 
    
    return result

In [378]:
def f(x, k):
    return np.e**(-x)*np.sin(k*x)

def fk(k):
    return lambda t: f(t, k)

def true_integral(k):
    return (k - np.e**(-np.pi)*(k*np.cos(k*np.pi)+ np.sin(k*np.pi)))/(1+k**2)

In [379]:
k = 10
npts = 1000

def get_result(k, npts):
    calc_value = simpson(fk(k), 0, np.pi, npts)
    analytic_value = true_integral(k)
    
    h = np.pi/(npts-1)
    theoretic_error = h**4*(k**4 + 4*k**3 - 6*k**2 - 4*k + 1)*np.pi/180
    
    I_2h = simpson(fk(k), 0, np.pi, int((npts+1)/2))
    runge_error = np.abs(I_2h - calc_value)/15
    
    print("     Simpson     ", calc_value)
    print(" Analytic answer ", analytic_value)
    print("Theoretic error  ", theoretic_error)
    print("    Runge error  ", runge_error)
    print("   Actual error  ", np.abs(calc_value - analytic_value))

interact(get_result, k=widgets.IntSlider(min=10, max=100, step=10, value=10), \
        npts=widgets.IntSlider(min=9, max=101, step=2, value=5))

interactive(children=(IntSlider(value=10, description='k', min=10, step=10), IntSlider(value=9, description='n…

<function __main__.get_result(k, npts)>

In [194]:
def approx_exp(n):
    t = smp.Symbol('t')

    x = np.linspace(0, np.pi, n+1)
    y = np.e**(-x)

    points = [(x[i], y[i]) for i in range(len(x))]

    return interpolate(points, t)

In [191]:
approx_exp(2)

0.127148919057585*t**2 - 0.704004578802886*t + 1.0

In [193]:
approx_exp(3)

-0.0396878858767694*t**3 + 0.316775260537909*t**2 - 0.908029567005286*t + 1.0

Погрешность интерполяции равна $\frac{M_{n+1}}{(n+1)!}\|w(x)\|_C, \quad w(x) = (x - x_0)(x - x_1)...(x - x_n)$

Тогда $\int_0^\pi R_n(x) \, sin \, kx \, dx < \frac{\pi M_{n+1}[e^{-x}]}{(n+1)!}\|w_n(x)\|_C, \quad R_n(x) = e^{-x} - L_n(x), \quad L_n - \text{аппроксимирующий многочлен}$

In [198]:
def omega(n):
    x = np.linspace(0, np.pi, n+1)

    t = smp.Symbol('t')

    expr = smp.S.One

    for i in range(len(x)):
        expr *= (t - x[i])

    return smp.expand(expr)

In [316]:
omega2 = omega(2)
omega3 = omega(3)

max2 = smp.calculus.util.maximum(omega2, t, smp.Interval(0, np.pi))
max3 = smp.calculus.util.maximum(omega3, t, smp.Interval(0, np.pi))
min2 = smp.calculus.util.minimum(omega2, t, smp.Interval(0, np.pi))
min3 = smp.calculus.util.minimum(omega3, t, smp.Interval(0, np.pi))

display(Math(f"\omega_2 = {smp.latex(omega2)}"))
display(Math(f"\omega_3 = {smp.latex(omega3)}"))
display(Math(f"\|\omega_2\| = {smp.latex(smp.Max(smp.Abs(max2), smp.Abs(min2)))}"))
display(Math(f"\|\omega_3\| = {smp.latex(smp.Max(smp.Abs(max3), smp.Abs(min3)))}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [282]:
def derivative(f, n):
    output = f
    
    for i in range(n):
        output = smp.diff(output, t)
        
    return output

In [317]:
func = smp.E**(-t)

der3 = derivative(func, 3)
der4 = derivative(func, 4)

max_der3 = smp.calculus.util.maximum(der3, t, smp.Interval(0, np.pi))
max_der4 = smp.calculus.util.maximum(der4, t, smp.Interval(0, np.pi))
min_der3 = smp.calculus.util.minimum(der3, t, smp.Interval(0, np.pi))
min_der4 = smp.calculus.util.minimum(der4, t, smp.Interval(0, np.pi))

M3 = smp.Max(smp.Abs(max_der3), smp.Abs(min_der3))
M4 = smp.Max(smp.Abs(max_der4), smp.Abs(min_der4))

display(Math(f"f^{{(3)}}= {smp.latex(der3)}; M_3 = {smp.latex(M3)}"))
display(Math(f"f^{{(4)}}= {smp.latex(der4)}; M_4 = {smp.latex(M4)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Погрешность для $n = 2: \; \frac{1.49179018232826}{4} \pi \approx 1.4$

Погрешность для $n = 3: \; \frac{1.20258137079018}{24} \pi \approx 0.16$

Пусть k - четное число, тогда 
$$
\int_0^\pi (ax^3 + bx^2 + cx + d) \, sin \, kx \, dx = \frac{\pi}{k} \left( \frac{6a}{k^2} - (a\pi^2 + b\pi + c) \right)
$$

$$ n = 2: a = 0, \; b = 0.127148919057585, \; c = 0.704004578802886, \; d = 1 $$

$$ n = 3: a = −0.0396878858767694, \; b = 0.316775260537909, \; c = −0.908029567005286, \; d = 1 $$

Можно получить более точную оценку погрешности $\frac{M_{n+1}[e^{-x}]}{(n+1)!} \int_0^\pi |\omega_n(x)| \, sin \, kx \, dx$

$$
\int_a^b P_4(x) \, sin \, kx \, dx = \left. sin \, kx \, \left(\frac{P_4^{(1)}(x)}{k^2} - \frac{P_4^{(3)}(x)}{k^4}\right) - cos \, kx \, \left(\frac{P_4(x)}{k} - \frac{P_4^{(2)}(x)}{k^3} + \frac{P_4^{(4)}(x)}{k^5}\right) \right|_{\,a}^{\,b}, \quad P_4(x) = ax^4 + bx^3 + cx^2 + dx + e
$$

$$
\int_0^\pi |\omega_2(x)| \, sin \, kx \, dx = 2 \int_0^{\frac{\pi}{2}} \omega_2(x) \, sin \, kx \, dx - \int_0^\pi \omega_2(x) \, sin \, kx \, dx
$$

$$
\int_0^\pi |\omega_3(x)| \, sin \, kx \, dx = 2 \int_\frac{\pi}{3}^{\frac{2\pi}{3}} \omega_3(x) \, sin \, kx \, dx - \int_0^\pi \omega_3(x) \, sin \, kx \, dx
$$

$$
\int_0^\pi \omega_2(x) \, sin \, kx \, dx = \frac{\pi}{k} \left( \frac{6}{k^2} - (\pi^2 −4.71238898038469\pi + 4.93480220054468) \right)
$$

$$
\int_0^\pi \omega_3(x) \, sin \, kx \, dx = \frac{\pi}{k}\left(-(\pi^3 - 6.28318530717959\pi^2 + 12.0628498235537\pi - 6.89028370673329) + \frac{12\pi - 37.6991118430775}{k^2} - \frac{24}{k^4}\right)
$$

In [388]:
def I(a, b, c, d, k):
    return np.pi*(6*a/k**2 - (a*np.pi**2 + b*np.pi + c))/k

a = 0
b = 0.127148919057585
c = 0.704004578802886
d = 1

def get_result(k):
    analytic_value = true_integral(k)
    
    print("Analytic answer", analytic_value)
    
    a = 0
    b = 0.127148919057585
    c = 0.704004578802886
    d = 1
    
    calc_value = I(a, b, c, d, k)
    
    omega2sin = np.pi*(6/k**2 - (np.pi**2 - 4.71238898038469*np.pi + 4.93480220054468))/k
    
    print()
    print("n = 2")
    print("Calculated answer", calc_value)
    print("Theoretic error", omega2sin/6)
    print("Actual error", np.abs(calc_value - analytic_value))
    
    a = -0.0396878858767694
    b = 0.316775260537909
    c = -0.908029567005286
    d = 1
    
    calc_value = I(a, b, c, d, k)
    
    omega3sin = np.pi*(-(np.pi**3 - 6.28318530717959*np.pi**2 + 12.0628498235537*np.pi - 6.89028370673329) + \
                      (12*np.pi - 37.6991118430775)/k**2 - 24/k**4)/k
    
    print()
    print("n = 3")
    print("Calculated answer", calc_value)
    print("Theoretic error", omega3sin/24)
    print("Actual error", np.abs(calc_value - analytic_value))
    
    
interact(get_result, k=widgets.IntSlider(min=10, max=100, step=10, value=10))

interactive(children=(IntSlider(value=10, description='k', min=10, step=10), Output()), _dom_classes=('widget-…

<function __main__.get_result(k)>

In [376]:
k = 100

np.pi*(6*-0.0396878858767694/k**3 - (-0.0396878858767694*np.pi**2 + 0.316775260537909*np.pi + -0.908029567005286)/k)

In [385]:
6 * -6.28318530717959